In [1]:
import gzip
import json
from pathlib import Path

import pandas as pd

DATA_DIR = Path("open")

# 1. 파일 존재 확인
assert (DATA_DIR / "dev.jsonl.gz").exists()
assert (DATA_DIR / "dev_labels.csv").exists()

# 2. dev 200건 읽기
with gzip.open(DATA_DIR / "dev.jsonl.gz", "rt", encoding="utf-8") as f:
    dev = [json.loads(line) for line in f]

# 3. 정답 읽기
labels = pd.read_csv(DATA_DIR / "dev_labels.csv")

# 4. 기본 구조 확인
print("공고 수:", len(dev))
print("정답 수:", len(labels))
print("공고 데이터 키:", dev[0].keys())
print("정답 열 개수:", len(labels.columns))

# 5. 개수 검증
assert len(dev) == 200
assert len(labels) == 200
assert len(labels.columns) == 49

# 6. ID 검증
dev_ids = [record["id"] for record in dev]
label_ids = labels["id"].tolist()

assert dev_ids == label_ids
assert len(dev_ids) == len(set(dev_ids))

print("ID 매칭 확인 완료")

# 7. 제출 컬럼 구조 검증
v_cols = [f"v{i}" for i in range(1, 25)]
e_cols = [f"e{i}" for i in range(1, 25)]

expected_cols = ["id"] + v_cols + e_cols

assert labels.columns.tolist() == expected_cols

print("49개 컬럼 구조 확인 완료")

# 8. 각 항목의 위반 건수
positive_counts = labels[v_cols].sum()

print("\n--- 항목별 위반 건수 ---")
print(positive_counts)

공고 수: 200
정답 수: 200
공고 데이터 키: dict_keys(['id', 'docs', 'dropped_doc_counts', 'input_completeness', 'assembly_policy_version', 'meta', 'anon_applied'])
정답 열 개수: 49
ID 매칭 확인 완료
49개 컬럼 구조 확인 완료

--- 항목별 위반 건수 ---
v1     7
v2     7
v3     8
v4     6
v5     7
v6     6
v7     7
v8     6
v9     6
v10    7
v11    6
v12    6
v13    6
v14    8
v15    6
v16    6
v17    6
v18    7
v19    6
v20    5
v21    6
v22    5
v23    5
v24    8
dtype: int64


In [3]:
print("최상위 키:", item_table.keys())

최상위 키: dict_keys(['기준일', '표기규칙', '부재탐지항목', '항목'])


In [4]:
items = item_table["항목"]

print("항목의 자료형:", type(items))
print("항목의 개수:", len(items))

if isinstance(items, dict):
    print("항목의 키:", items.keys())
elif isinstance(items, list):
    print("첫 번째 항목:")
    print(json.dumps(items[0], ensure_ascii=False, indent=2))

항목의 자료형: <class 'dict'>
항목의 개수: 24
항목의 키: dict_keys(['v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9', 'v10', 'v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19', 'v20', 'v21', 'v22', 'v23', 'v24'])


In [5]:
print("--- v1 공식 정의 ---")
print(json.dumps(items["v1"], ensure_ascii=False, indent=2))

--- v1 공식 정의 ---
{
  "항목명": "참가자격 특정기관 제한",
  "부재탐지": false,
  "국가계약법": "국가계약법 시행령 제12조 국가계약법 시행령 제21조",
  "지방계약법": "지방계약법 시행령 제13조 지방계약법 시행령 제20조",
  "비고": ""
}


In [6]:
# v1 위반으로 정답이 붙은 dev 사례만 추출
v1_positive = labels.loc[
    labels["v1"] == 1,
    ["id", "v1", "e1"]
].copy()

print("v1 위반 건수:", len(v1_positive))
display(v1_positive)

v1 위반 건수: 7


,id,v1,e1
0,PPS-DEV-01,1,라. 본 용역은 「고등교육법」 제2조에 따른 대학 또는 「산업교육진흥\n및 산학연협...
34,PPS-DEV-035,1,NaN
45,PPS-DEV-046,1,NaN
48,PPS-DEV-049,1,NaN
57,PPS-DEV-058,1,NaN
62,PPS-DEV-063,1,NaN
71,PPS-DEV-072,1,NaN


In [8]:
# id -> dev 공고 빠르게 찾기 위한 인덱스
dev_by_id = {record["id"]: record for record in dev}


def inspect_case(case_id, violation="v1"):
    """dev의 특정 사례를 분석하기 쉽게 출력한다."""
    
    record = dev_by_id[case_id]
    row = labels.loc[labels["id"] == case_id].iloc[0]

    evidence_col = "e" + violation[1:]

    print("=" * 80)
    print("ID:", case_id)
    print("항목:", violation)
    print("정답:", row[violation])
    print("Evidence:", row[evidence_col])
    
    print("\n[meta]")
    print(json.dumps(record.get("meta", {}), ensure_ascii=False, indent=2))

    print("\n[문서 목록]")
    for i, doc in enumerate(record.get("docs", [])):
        print(
            f"{i}: "
            f"type={doc.get('type')} / "
            f"doc_id={doc.get('doc_id')} / "
            f"길이={len(doc.get('text', '')):,}"
        )

    print("\n[dropped_doc_counts]")
    print(record.get("dropped_doc_counts"))

    print("\n[input_completeness]")
    print(
        json.dumps(
            record.get("input_completeness", {}),
            ensure_ascii=False,
            indent=2
        )
    )

In [10]:
print("dev 첫 번째 ID:", repr(dev[0]["id"]))
print("labels 첫 번째 ID:", repr(labels.iloc[0]["id"]))

print("\ndev_by_id 첫 5개:")
print(list(dev_by_id.keys())[:5])

dev 첫 번째 ID: 'PPS-DEV-01'
labels 첫 번째 ID: 'PPS-DEV-01'

dev_by_id 첫 5개:
['PPS-DEV-01', 'PPS-DEV-02', 'PPS-DEV-03', 'PPS-DEV-04', 'PPS-DEV-05']


In [11]:
case_id = v1_positive.iloc[0]["id"]

print("분석할 ID:", repr(case_id))
inspect_case(case_id, "v1")

분석할 ID: 'PPS-DEV-01'
ID: PPS-DEV-01
항목: v1
정답: 1
Evidence: 라. 본 용역은 「고등교육법」 제2조에 따른 대학 또는 「산업교육진흥
및 산학연협력촉진에 관한 법률」 제25조에 따른 산학협력단만 참여 가능함

[meta]
{
  "적용계약법": "지방계약법",
  "업무구분": "일반용역",
  "계약방법": "제한경쟁",
  "낙찰방법": "협상에의한계약",
  "낙찰하한율": null,
  "배정예산금액": 310000000,
  "입찰추정가격": 281818182,
  "소관구분": "지방공기업",
  "공동도급구성방식": "공동이행",
  "정보화사업여부": "미입력",
  "세부품명번호목록": null,
  "제한지역코드목록": "부산광역시",
  "지역제한여부": "Y",
  "면허업종제한목록": "[기타자유업(행사대행업)(9901)]",
  "업종제한여부": "Y",
  "조항호내용": "[판로지원법 시행령] 중기업,소기업,소상공인 제한",
  "공고게시일자": "20260203",
  "개찰예정일자": "20260225",
  "긴급공고여부": "Y",
  "입찰방법": "전자입찰",
  "조달방식": "자체조달"
}

[문서 목록]
0: type=공고문 / doc_id=D0 / 길이=6,749
1: type=과업지시서 / doc_id=D2 / 길이=14,596

[dropped_doc_counts]
{'제안요청서': 1}

[input_completeness]
{
  "공고문_실재": true,
  "추출_성공": true,
  "무탈락": false,
  "완전관측": false
}


In [13]:
def summarize_positive_cases(violation):
    evidence_col = "e" + violation[1:]
    positive_rows = labels.loc[labels[violation] == 1]

    results = []

    for _, row in positive_rows.iterrows():
        case_id = row["id"]
        record = dev_by_id[case_id]
        meta = record.get("meta", {})

        results.append({
            "id": case_id,
            "evidence": row[evidence_col],
            "적용계약법": meta.get("적용계약법"),
            "계약방법": meta.get("계약방법"),
            "낙찰방법": meta.get("낙찰방법"),
            "배정예산금액": meta.get("배정예산금액"),
            "입찰추정가격": meta.get("입찰추정가격"),
            "지역제한여부": meta.get("지역제한여부"),
            "업종제한여부": meta.get("업종제한여부"),
            "문서수": len(record.get("docs", [])),
            "dropped_docs": record.get("dropped_doc_counts"),
        })

    return pd.DataFrame(results)


v1_summary = summarize_positive_cases("v1")

display(v1_summary)

,id,evidence,적용계약법,계약방법,낙찰방법,배정예산금액,입찰추정가격,지역제한여부,업종제한여부,문서수,dropped_docs
0,PPS-DEV-01,라. 본 용역은 「고등교육법」 제2조에 따른 대학 또는 「산업교육진흥\n및 산학연협...,지방계약법,제한경쟁,협상에의한계약,310000000,281818182,Y,Y,2,{'제안요청서': 1}
1,PPS-DEV-035,NaN,국가계약법,제한경쟁,협상에의한계약,60000000,60000000,N,N,1,{}
2,PPS-DEV-046,NaN,지방계약법,제한경쟁,적격심사제,610838800,555308000,Y,N,2,{}
3,PPS-DEV-049,NaN,지방계약법,제한경쟁,규격가격동시입찰,660000000,600000000,Y,Y,1,{}
4,PPS-DEV-058,NaN,국가계약법,제한경쟁,협상에의한계약,190000000,172727273,N,Y,1,{}
5,PPS-DEV-063,NaN,국가계약법,제한경쟁,적격심사제,182456000,165869091,Y,Y,1,{}
6,PPS-DEV-072,NaN,지방계약법,제한경쟁,규격가격동시입찰,42460000,38600000,Y,N,1,{}


In [14]:
def search_docs(case_id, keywords, context=150):
    record = dev_by_id[case_id]

    print("=" * 100)
    print("ID:", case_id)

    found = False

    for doc in record.get("docs", []):
        text = doc.get("text", "")

        for keyword in keywords:
            start = 0

            while True:
                pos = text.find(keyword, start)

                if pos == -1:
                    break

                found = True

                left = max(0, pos - context)
                right = min(len(text), pos + len(keyword) + context)

                print(f"\n[문서 type={doc.get('type')} / keyword={keyword}]")
                print(text[left:right].replace("\n", " "))

                start = pos + len(keyword)

    if not found:
        print("검색어가 발견되지 않았습니다.")

In [15]:
case_id = v1_positive.iloc[1]["id"]

search_docs(
    case_id,
    keywords=[
        "참가자격",
        "입찰참가",
        "참여자격",
        "참여 가능",
        "참여가능",
        "자격요건",
    ],
    context=200
)

ID: PPS-DEV-035

[문서 type=공고문 / keyword=참가자격]
직접 제출) 제출장소 : [수요기관(공공시설)] 관정관 [부서] 기술평가 실시 : 2026. 2월 중 / 관정관 5층 고광선회의실 (세부일시 확정 후 제안서 제출업체 개별통보) 협상 실시 : 협상업체 개별통보 낙찰자 통보 및 계약체결 : 낙찰업체 개별통보 사. 기타 : 세부사항은 제안요청서와 같으며, 사업금액 및 물량은 증감·조정될 수 있습니다. 2. 입찰참가자격 가. 「국가를 당사자로 하는 계약에 관한 법률 시행령(2026.1.2.)」 제12조 및 동법 시행규칙(2026.1.2.) 제14조에 따른 자격요건을 갖춘 사업자 - 사업자 등록증 상에 서적(도서) 도·소매, 무역(도서) 등이 명시되어 있는 업체로서, 국가종합전자조달시스템 입찰참가자격등록규정에 의하여 반드시 입찰 공고에서 정한 기한 내에 G2B 물품분류번호

[문서 type=공고문 / keyword=참가자격]
요청서와 같으며, 사업금액 및 물량은 증감·조정될 수 있습니다. 2. 입찰참가자격 가. 「국가를 당사자로 하는 계약에 관한 법률 시행령(2026.1.2.)」 제12조 및 동법 시행규칙(2026.1.2.) 제14조에 따른 자격요건을 갖춘 사업자 - 사업자 등록증 상에 서적(도서) 도·소매, 무역(도서) 등이 명시되어 있는 업체로서, 국가종합전자조달시스템 입찰참가자격등록규정에 의하여 반드시 입찰 공고에서 정한 기한 내에 G2B 물품분류번호 5510151001(서적)을 등록한 사업자 나. 「중소기업기본법」 제2조제2항에 따른 중소기업 또는 「소상공인 보호 및 지원에 관한 법률」 제2조에 따른 소상공인으로서 중소기업 또는 소상공인확인서(전자입찰서 제출 마감일 전일까지 발급된 것으로 유효기간 내에 있어야 함)를 소지한 업체

[문서 type=공고문 / keyword=참가자격]
공인확인서(전자입찰서 제출 마감일 전일까지 발급된 것으로 유효기간 내에 있어야 함)를 소지한 업체 ※ 위의 조건을 모두 갖춘 사업자의 경우에만 입찰에 

In [17]:
def extract_section(case_id, start_keyword, end_keywords):
    record = dev_by_id[case_id]

    print("=" * 100)
    print("ID:", case_id)

    for doc in record.get("docs", []):
        text = doc.get("text", "")

        start = text.find(start_keyword)

        if start == -1:
            continue

        # 시작점 이후에 등장하는 다음 절의 위치 찾기
        candidates = []

        for end_keyword in end_keywords:
            pos = text.find(end_keyword, start + len(start_keyword))

            if pos != -1:
                candidates.append(pos)

        end = min(candidates) if candidates else min(len(text), start + 5000)

        print(f"\n[문서 type={doc.get('type')}]")
        print(text[start:end])


case_id = v1_positive.iloc[1]["id"]

extract_section(
    case_id,
    start_keyword="2. 입찰참가자격",
    end_keywords=[
        "3.",
        "Ⅲ.",
        "III."
    ]
)

ID: PPS-DEV-035

[문서 type=공고문]
2. 입찰참가자격
가. 「국가를 당사자로 하는 계약에 관한 법률 시행령(2026.1.2.)」 제12조 및 동법 시행규칙(2026.1.2.) 제14조에 따른 자격요건을 갖춘 사업자
- 사업자 등록증 상에 서적(도서) 도·소매, 무역(도서) 등이 명시되어 있는 업체로서, 국가종합전자조달시스템 입찰참가자격등록규정에 의하여 반드시 입찰 공고에서 정한 기한 내에 G2B 물품분류번호 5510151001(서적)을 등록한 사업자
나. 「중소기업기본법」 제2조제2항에 따른 중소기업 또는 「소상공인 보호 및 지원에 관한 법률」 제2조에 따른 소상공인으로서 중소기업 또는 소상공인확인서(전자입찰서 제출 마감일 전일까지 발급된 것으로 유효기간 내에 있어야 함)를 소지한 업체
※ 위의 조건을 모두 갖춘 사업자의 경우에만 입찰에 참여할 수 있음
다. 본 입찰은 4년제 대학교만 참여 가능
라. 「국가를 당사자로 하는 계약에 관한 법률(2026.1.2.)」 제27조 및 동법 시행령 (2026.1.2.) 제76조에 의한 부정당업자로서 입찰참가자격의 제한을 받지 않은 업체이어야 합니다.
마. 납품(공급)업체의 경우 전자입찰서 제출 마감일 전일까지 제조사로부터 당해 물품공급 및 무상지원(A/S) 확약서를 발급후 보유 및 제출하여야 합니다.
바. 공동 수급 및 하도급 불허
사. [기관(대학)] 퇴직자가 설립한 업체 및 임원급 이상으로 재취업한 업체는 입찰에 참여할 수 없습니다.(단, 퇴직자의 설립일 및 재취업일로부터 2년 이내)
※ 무자격자가 고의로 입찰에 참가하여 「국가를 당사자로 하는 계약에 관한 법률 시행령」 제76조 제2항의 각 호에 해당된다고 판단될 경우에는 관계규정에 따라 부정당업자로 제재할 수 있습니다.



In [18]:
LAW_DIR = DATA_DIR / "data" / "법령패키지" / "법령"

law_files = sorted(LAW_DIR.glob("*.txt"))

print("법령 파일 수:", len(law_files))

for path in law_files:
    print(path.name)

법령 파일 수: 23
(계약예규) 공동계약운용요령.txt
(계약예규) 정부 입찰·계약 집행기준.txt
국가를 당사자로 하는 계약에 관한 법률 등의 재정경제부장관이 정하는 고시금액.txt
국가를 당사자로 하는 계약에 관한 법률 시행규칙.txt
국가를 당사자로 하는 계약에 관한 법률 시행령.txt
국가를 당사자로 하는 계약에 관한 법률.txt
소상공인기본법 시행령.txt
소상공인기본법.txt
소프트웨어 진흥법 시행령.txt
소프트웨어 진흥법.txt
중소 소프트웨어사업자의 사업 참여 지원에 관한 지침.txt
중소기업기본법 시행령.txt
중소기업기본법.txt
중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역.txt
중소기업자간 경쟁제품 직접생산 확인기준.txt
중소기업제품 구매촉진 및 판로지원에 관한 법률 시행규칙.txt
중소기업제품 구매촉진 및 판로지원에 관한 법률 시행령.txt
중소기업제품 구매촉진 및 판로지원에 관한 법률.txt
지방자치단체 입찰 및 계약 집행기준.txt
지방자치단체 입찰시 낙찰자 결정기준.txt
지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙.txt
지방자치단체를 당사자로 하는 계약에 관한 법률 시행령.txt
지방자치단체를 당사자로 하는 계약에 관한 법률.txt


In [20]:
import re

def extract_article(file_path, article_number):
    """
    법령 txt 파일에서 특정 조(예: 제12조)를 찾아
    다음 조가 시작되기 전까지 반환한다.
    """
    text = file_path.read_text(encoding="utf-8")

    # 예: 제12조, 제12조의2 모두 고려
    start_pattern = rf"제{article_number}조(?:의\d+)?\s*\("
    start_match = re.search(start_pattern, text)

    if start_match is None:
        print(f"제{article_number}조를 찾지 못했습니다.")
        return None

    start = start_match.start()

    # 현재 조 다음에 나오는 '제N조'를 찾는다.
    next_match = re.search(
        r"\n제\d+조(?:의\d+)?\s*\(",
        text[start_match.end():]
    )

    if next_match:
        end = start_match.end() + next_match.start()
    else:
        end = len(text)

    return text[start:end].strip()

In [21]:
national_decree = (
    LAW_DIR / "국가를 당사자로 하는 계약에 관한 법률 시행령.txt"
)

local_decree = (
    LAW_DIR / "지방자치단체를 당사자로 하는 계약에 관한 법률 시행령.txt"
)

In [22]:
print("===== 국가계약법 시행령 제12조 =====")
print(extract_article(national_decree, 12))

print("\n===== 국가계약법 시행령 제21조 =====")
print(extract_article(national_decree, 21))

print("\n===== 지방계약법 시행령 제13조 =====")
print(extract_article(local_decree, 13))

print("\n===== 지방계약법 시행령 제20조 =====")
print(extract_article(local_decree, 20))

===== 국가계약법 시행령 제12조 =====
제12조(경쟁입찰의 참가자격)
  ①각 중앙관서의 장 또는 계약담당공무원은 다음 각호의 요건을 갖춘 자에 한하여 경쟁입찰에 참가하게 하여야 한다. <개정 1996.12.31, 1999.9.9, 2008.2.29, 2025.12.30>
    1. 삭제<1999.9.9>
    2. 다른 법령의 규정에 의하여 허가ㆍ인가ㆍ면허ㆍ등록ㆍ신고등을 요하거나 자격요건을 갖추어야 할 경우에는 당해 허가ㆍ인가ㆍ면허ㆍ등록ㆍ신고등을 받았거나 당해 자격요건에 적합할 것
    3. 보안측정등의 조사가 필요한 경우에는 관계기관으로부터 적합판정을 받을 것
    4. 기타 재정경제부령이 정하는 요건에 적합할 것
  ②「중소기업협동조합법」에 따른 중소기업협동조합이 물품의 제조ㆍ구매ㆍ임차 또는 용역에 관한 경쟁입찰에 참가하는 경우(제1항제2호에 따른 요건을 갖춘 조합원으로 하여금 해당 물품을 제조ㆍ공급 또는 임대하게 하거나 용역을 수행하게 하는 경우로 한정한다)에는 제1항제2호를 적용하지 아니한다. <신설 1999.9.9, 2005.9.8, 2006.5.30, 2007.10.10, 2013.6.17, 2024.12.24>
    1. 삭제<2007.10.10>
    2. 삭제<2007.10.10>
  ③ 법 제27조의5제1항에서 "대통령령으로 정하는 조세포탈 등을 한 자"란 다음 각 호의 어느 하나에 해당하는 자를 말한다. <신설 2013.12.30, 2017.3.27, 2018.12.4, 2019.9.17, 2021.2.17>
    1. 「조세범 처벌법」 제3조에 따른 조세 포탈세액이나 환급ㆍ공제받은 세액이 5억원 이상인 자
    2. 「관세법」 제270조에 따른 부정한 방법으로 관세를 감면받거나 면탈하거나 환급받은 세액이 5억원 이상인 자
    3. 「지방세기본법」 제102조에 따른 지방세 포탈세액이나 환급ㆍ공제 세액이 5억원 이상인 자
    4. 「국제조세조정에 관한 법률」 제53조에 따른 해외금융계좌의 신고의무를 위반하고, 

In [23]:
import gzip
import json
import re
from pathlib import Path

import pandas as pd


# ============================================================
# 0. 경로
# ============================================================

DATA_DIR = Path("open")
LAW_DIR = DATA_DIR / "data" / "법령패키지" / "법령"

DEV_PATH = DATA_DIR / "dev.jsonl.gz"
LABEL_PATH = DATA_DIR / "dev_labels.csv"
ITEM_PATH = DATA_DIR / "data" / "항목표.json"


# ============================================================
# 1. 데이터 로드
# ============================================================

assert DEV_PATH.exists(), f"파일 없음: {DEV_PATH}"
assert LABEL_PATH.exists(), f"파일 없음: {LABEL_PATH}"
assert ITEM_PATH.exists(), f"파일 없음: {ITEM_PATH}"
assert LAW_DIR.exists(), f"폴더 없음: {LAW_DIR}"

with gzip.open(DEV_PATH, "rt", encoding="utf-8") as f:
    dev = [json.loads(line) for line in f]

labels = pd.read_csv(LABEL_PATH)

with open(ITEM_PATH, "r", encoding="utf-8") as f:
    item_table = json.load(f)

items = item_table["항목"]

v_cols = [f"v{i}" for i in range(1, 25)]
e_cols = [f"e{i}" for i in range(1, 25)]

dev_by_id = {
    record["id"]: record
    for record in dev
}


# ============================================================
# 2. 기본 검증
# ============================================================

assert len(dev) == 200
assert len(labels) == 200
assert len(labels.columns) == 49
assert len(items) == 24

expected_cols = ["id"] + v_cols + e_cols
assert labels.columns.tolist() == expected_cols

dev_ids = [record["id"] for record in dev]
label_ids = labels["id"].tolist()

assert dev_ids == label_ids
assert len(dev_ids) == len(set(dev_ids))

print("[1] 데이터 기본 검증 완료")
print(f"    dev: {len(dev)}건")
print(f"    labels: {len(labels)}건")
print(f"    심사항목: {len(items)}개")
print(f"    법령파일: {len(list(LAW_DIR.glob('*.txt')))}개")


# ============================================================
# 3. 24개 항목 요약표
# ============================================================

summary_rows = []

for v in v_cols:
    e = "e" + v[1:]

    positive_mask = labels[v] == 1

    summary_rows.append({
        "항목": v,
        "항목명": items[v].get("항목명"),
        "부재탐지": items[v].get("부재탐지"),
        "양성수": int(positive_mask.sum()),
        "evidence_있음": int(
            labels.loc[positive_mask, e].notna().sum()
        ),
        "국가계약법": items[v].get("국가계약법"),
        "지방계약법": items[v].get("지방계약법"),
    })

item_summary = pd.DataFrame(summary_rows)

print("\n[2] 24개 항목 요약")
display(item_summary)


# ============================================================
# 4. 특정 항목의 양성 사례 요약
# ============================================================

def get_positive_cases(violation):
    """
    예:
        get_positive_cases("v1")
    """

    if violation not in v_cols:
        raise ValueError(f"잘못된 항목: {violation}")

    evidence_col = "e" + violation[1:]
    positive_rows = labels.loc[labels[violation] == 1]

    results = []

    for _, row in positive_rows.iterrows():

        case_id = row["id"]
        record = dev_by_id[case_id]
        meta = record.get("meta", {})

        results.append({
            "id": case_id,
            "evidence": row[evidence_col],
            "적용계약법": meta.get("적용계약법"),
            "업무구분": meta.get("업무구분"),
            "계약방법": meta.get("계약방법"),
            "낙찰방법": meta.get("낙찰방법"),
            "배정예산금액": meta.get("배정예산금액"),
            "입찰추정가격": meta.get("입찰추정가격"),
            "지역제한여부": meta.get("지역제한여부"),
            "업종제한여부": meta.get("업종제한여부"),
            "문서수": len(record.get("docs", [])),
            "dropped_docs": record.get("dropped_doc_counts"),
        })

    return pd.DataFrame(results)


# ============================================================
# 5. 특정 사례 기본 정보
# ============================================================

def inspect_case(case_id, violation=None):
    """
    공고 하나의 핵심 구조 확인
    """

    if case_id not in dev_by_id:
        raise KeyError(f"존재하지 않는 ID: {case_id}")

    record = dev_by_id[case_id]

    print("=" * 80)
    print("ID:", case_id)

    if violation is not None:

        evidence_col = "e" + violation[1:]

        row = labels.loc[
            labels["id"] == case_id
        ].iloc[0]

        print("항목:", violation)
        print("정답:", row[violation])
        print("Evidence:", row[evidence_col])

    print("\n[meta]")
    print(
        json.dumps(
            record.get("meta", {}),
            ensure_ascii=False,
            indent=2
        )
    )

    print("\n[문서 목록]")

    for i, doc in enumerate(record.get("docs", [])):

        print(
            f"{i}: "
            f"type={doc.get('type')} / "
            f"doc_id={doc.get('doc_id')} / "
            f"길이={len(doc.get('text', '')):,}"
        )

    print("\n[dropped_doc_counts]")
    print(record.get("dropped_doc_counts"))

    print("\n[input_completeness]")
    print(
        json.dumps(
            record.get("input_completeness", {}),
            ensure_ascii=False,
            indent=2
        )
    )


# ============================================================
# 6. 문서 내 검색
# ============================================================

def search_docs(case_id, keywords, context=200):
    """
    특정 공고의 모든 docs에서 키워드를 검색하고
    앞뒤 문맥을 출력한다.
    """

    if case_id not in dev_by_id:
        raise KeyError(f"존재하지 않는 ID: {case_id}")

    record = dev_by_id[case_id]

    print("=" * 100)
    print("ID:", case_id)

    found = False

    for doc in record.get("docs", []):

        text = doc.get("text", "")

        for keyword in keywords:

            start = 0

            while True:

                pos = text.find(keyword, start)

                if pos == -1:
                    break

                found = True

                left = max(0, pos - context)
                right = min(
                    len(text),
                    pos + len(keyword) + context
                )

                print(
                    f"\n[문서={doc.get('type')} / "
                    f"검색어={keyword}]"
                )

                print(
                    text[left:right]
                    .replace("\n", " ")
                )

                start = pos + len(keyword)

    if not found:
        print("검색 결과 없음")


# ============================================================
# 7. Evidence가 실제 어느 문서에 있는지 찾기
# ============================================================

def find_evidence(case_id, violation):
    """
    정답 evidence가 존재한다면
    실제 docs의 어느 문서에 있는지 확인한다.
    """

    evidence_col = "e" + violation[1:]

    row = labels.loc[
        labels["id"] == case_id
    ].iloc[0]

    evidence = row[evidence_col]

    if pd.isna(evidence):
        print("정답 evidence가 비어 있습니다.")
        return

    record = dev_by_id[case_id]

    found = False

    for doc in record.get("docs", []):

        text = doc.get("text", "")

        pos = text.find(str(evidence))

        if pos != -1:

            found = True

            print("=" * 80)
            print("문서 type:", doc.get("type"))
            print("doc_id:", doc.get("doc_id"))
            print()

            left = max(0, pos - 300)
            right = min(
                len(text),
                pos + len(str(evidence)) + 300
            )

            print(text[left:right])

    if not found:
        print("Evidence를 docs에서 찾지 못했습니다.")


# ============================================================
# 8. 법령 조문 추출
# ============================================================

def extract_article(file_path, article_number):
    """
    법령 txt에서 특정 조를 추출한다.
    예:
        extract_article(path, 21)
    """

    text = file_path.read_text(encoding="utf-8")

    pattern = rf"제{article_number}조(?:의\d+)?\s*\("

    start_match = re.search(pattern, text)

    if start_match is None:
        return None

    start = start_match.start()

    next_match = re.search(
        r"\n제\d+조(?:의\d+)?\s*\(",
        text[start_match.end():]
    )

    if next_match:
        end = (
            start_match.end()
            + next_match.start()
        )
    else:
        end = len(text)

    return text[start:end].strip()


# ============================================================
# 9. 법령 파일 검색
# ============================================================

def find_law_files(keyword):
    """
    파일명에 keyword가 들어가는 법령 파일 검색
    """

    matches = []

    for path in sorted(LAW_DIR.glob("*.txt")):
        if keyword in path.name:
            matches.append(path)

    return matches


print("\n[3] 분석 함수 준비 완료")

[1] 데이터 기본 검증 완료
    dev: 200건
    labels: 200건
    심사항목: 24개
    법령파일: 23개

[2] 24개 항목 요약


,항목,항목명,부재탐지,양성수,evidence_있음,국가계약법,지방계약법
0,v1,참가자격 특정기관 제한,False,7,1,국가계약법 시행령 제12조 국가계약법 시행령 제21조,지방계약법 시행령 제13조 지방계약법 시행령 제20조
1,v2,고시금액 미만 실적제한,False,7,7,국가계약법 시행령 제21조 제1항 국가계약법 시행규칙 제25조,지방계약법 시행령 제20조 제1항 지방계약 법시행규칙제25조 제2항 지방자치단체 입...
2,v3,실적제한 1배수 이상,False,8,8,국가계약법 시행령 제21조 제1항 국가계약법 시행규칙 제25조 (계약예규) 정부 입...,지방계약법 시행령 제20조 제1항 지방계약법 시행규칙 제25조
3,v4,"고시금액 이상 특정기관, 특정실적",False,6,5,(계약예규) 정부입찰계약집행기준 제2장 제한경쟁입찰의 운용 제5조,(행안부예규) 지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약 일반기준
4,v5,고시금액 이상 지역제한,False,7,1,국가계약법 시행령 제21조 국가를 당사자로하는 계약에 관한 법률 등의 재정경제부장관...,지방계약법 시행령 제20조 지방계약법 시행규칙 제24조
5,v6,"고시금액 미만 지역제한 시,군,구",False,6,1,국가계약법 시행규칙 제25조,지방계약법 시행규칙 제25조 지방자치단체 입찰 및 계약집행기준 제5장 수의계약 운영...
6,v7,고시금액 미만 지역제한 인접 확대,False,7,7,국가계약법 시행규칙 제25조,지방계약법 시행규칙 제25조 지방자치단체 입찰 및 계약집행기준 제5장 수의계약 운영...
7,v8,중복제한 (실적+지역),False,6,2,국가계약법 시행규칙 제25조,지방계약법 시행규칙 제25조 지방자치단체 입찰 및 계약집행기준 제5장 수의계약 운영...
8,v9,과업지시서 특정 모델명 명시,False,6,3,(계약예규) 정부 입찰·계약 집행기준 제2장 제한경쟁입찰의 운용 제5조,(행안부예규) 지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약일반기준 7....
9,v10,중기간 경쟁제품 입찰 직생 없음,True,7,0,중소기업제품 구매촉진 및 판로지원에 관한 법률 제9조 중소기업자간 경쟁제품 및 공사...,중소기업제품 구매촉진 및 판로지원에 관한 법률 제9조 중소기업자간 경쟁제품 및 공사...



[3] 분석 함수 준비 완료


# 24개 항목 전체 구조 분석

In [24]:
# ============================================================
# 10. 24개 항목 전체 구조 분석
# ============================================================

analysis_rows = []

for v in v_cols:
    info = items[v]
    e = "e" + v[1:]

    positive = labels[labels[v] == 1]

    # 양성 중 evidence가 실제로 제공된 개수
    evidence_count = int(positive[e].notna().sum())

    # 양성 사례들의 계약법 종류
    contract_laws = set()

    # 양성 사례들의 문서 type
    doc_types = set()

    # 문서 누락이 존재하는 양성 사례 수
    dropped_count = 0

    for case_id in positive["id"]:

        record = dev_by_id[case_id]
        meta = record.get("meta", {})

        law = meta.get("적용계약법")

        if law is not None:
            contract_laws.add(str(law))

        for doc in record.get("docs", []):
            doc_type = doc.get("type")

            if doc_type:
                doc_types.add(doc_type)

        dropped = record.get("dropped_doc_counts")

        if dropped:
            dropped_count += 1

    analysis_rows.append({
        "항목": v,
        "항목명": info.get("항목명"),
        "부재탐지": info.get("부재탐지"),
        "양성수": len(positive),
        "Evidence수": evidence_count,
        "적용계약법": ", ".join(sorted(contract_laws)),
        "등장문서종류": ", ".join(sorted(doc_types)),
        "문서누락_양성수": dropped_count,
        "국가계약법근거": info.get("국가계약법"),
        "지방계약법근거": info.get("지방계약법"),
        "비고": info.get("비고"),
    })

analysis_table = pd.DataFrame(analysis_rows)

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 100)

display(analysis_table)

,항목,항목명,부재탐지,양성수,Evidence수,적용계약법,등장문서종류,문서누락_양성수,국가계약법근거,지방계약법근거,비고
0,v1,참가자격 특정기관 제한,False,7,1,"국가계약법, 지방계약법","공고문, 과업지시서, 규격서",1,국가계약법 시행령 제12조 국가계약법 시행령 제21조,지방계약법 시행령 제13조 지방계약법 시행령 제20조,
1,v2,고시금액 미만 실적제한,False,7,7,"국가계약법, 지방계약법","공고문, 과업지시서",1,국가계약법 시행령 제21조 제1항 국가계약법 시행규칙 제25조,지방계약법 시행령 제20조 제1항 지방계약 법시행규칙제25조 제2항 지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약 일반기준 제1절 총칙 7. 계약담당자 주의사항,지방 + 소액수의 가능
2,v3,실적제한 1배수 이상,False,8,8,"국가계약법, 지방계약법","공고문, 과업지시서, 제안요청서",0,국가계약법 시행령 제21조 제1항 국가계약법 시행규칙 제25조 (계약예규) 정부 입찰·계약 집행기준 제2장 제한경쟁입찰의 운용 제5조,지방계약법 시행령 제20조 제1항 지방계약법 시행규칙 제25조,사업예산 기준
3,v4,"고시금액 이상 특정기관, 특정실적",False,6,5,"국가계약법, 지방계약법","공고문, 제안요청서",0,(계약예규) 정부입찰계약집행기준 제2장 제한경쟁입찰의 운용 제5조,(행안부예규) 지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약 일반기준,특정기관 표현 다양
4,v5,고시금액 이상 지역제한,False,7,1,"국가계약법, 지방계약법","공고문, 규격서, 제안요청서",0,국가계약법 시행령 제21조 국가를 당사자로하는 계약에 관한 법률 등의 재정경제부장관이 정하는 고시금액,지방계약법 시행령 제20조 지방계약법 시행규칙 제24조,"지방, 지자체에 따라 고시금액 다름"
5,v6,"고시금액 미만 지역제한 시,군,구",False,6,1,"국가계약법, 지방계약법","공고문, 제안요청서",0,국가계약법 시행규칙 제25조,지방계약법 시행규칙 제25조 지방자치단체 입찰 및 계약집행기준 제5장 수의계약 운영요령 제3절 수의계약 대상과 운영요령 1. 금액기준에 따른 2인 이상 견적서 제출 수의계약...,지방 + 소액수의 가능
6,v7,고시금액 미만 지역제한 인접 확대,False,7,7,"국가계약법, 지방계약법","공고문, 과업지시서, 규격서",0,국가계약법 시행규칙 제25조,지방계약법 시행규칙 제25조 지방자치단체 입찰 및 계약집행기준 제5장 수의계약 운영요령 제3절 수의계약 대상과 운영요령 1. 금액기준에 따른 2인 이상 견적서 제출 수의계약...,지방 + 소액수의 가능
7,v8,중복제한 (실적+지역),False,6,2,지방계약법,"공고문, 과업지시서, 제안요청서",0,국가계약법 시행규칙 제25조,지방계약법 시행규칙 제25조 지방자치단체 입찰 및 계약집행기준 제5장 수의계약 운영요령 제3절 수의계약 대상과 운영요령 1. 금액기준에 따른 2인 이상 견적서 제출 수의계약...,지방 + 소액수의 가능
8,v9,과업지시서 특정 모델명 명시,False,6,3,"국가계약법, 지방계약법","공고문, 규격서",0,(계약예규) 정부 입찰·계약 집행기준 제2장 제한경쟁입찰의 운용 제5조,(행안부예규) 지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약일반기준 7. 계약담당자 주의사항,
9,v10,중기간 경쟁제품 입찰 직생 없음,True,7,0,"국가계약법, 지방계약법","공고문, 과업지시서, 규격서",1,중소기업제품 구매촉진 및 판로지원에 관한 법률 제9조 중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역,중소기업제품 구매촉진 및 판로지원에 관한 법률 제9조 중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역,


In [25]:
# ============================================================
# 11. 항목별 양성 / Evidence 현황
# ============================================================

evidence_stats = []

for v in v_cols:
    e = "e" + v[1:]

    positive_mask = labels[v] == 1

    positives = int(positive_mask.sum())
    evidences = int(
        labels.loc[positive_mask, e].notna().sum()
    )

    evidence_stats.append({
        "항목": v,
        "항목명": items[v].get("항목명"),
        "부재탐지": items[v].get("부재탐지"),
        "양성": positives,
        "Evidence 있음": evidences,
        "Evidence 없음": positives - evidences,
    })

evidence_stats = pd.DataFrame(evidence_stats)

display(evidence_stats)

,항목,항목명,부재탐지,양성,Evidence 있음,Evidence 없음
0,v1,참가자격 특정기관 제한,False,7,1,6
1,v2,고시금액 미만 실적제한,False,7,7,0
2,v3,실적제한 1배수 이상,False,8,8,0
3,v4,"고시금액 이상 특정기관, 특정실적",False,6,5,1
4,v5,고시금액 이상 지역제한,False,7,1,6
5,v6,"고시금액 미만 지역제한 시,군,구",False,6,1,5
6,v7,고시금액 미만 지역제한 인접 확대,False,7,7,0
7,v8,중복제한 (실적+지역),False,6,2,4
8,v9,과업지시서 특정 모델명 명시,False,6,3,3
9,v10,중기간 경쟁제품 입찰 직생 없음,True,7,0,7


# 24개 항목 판정 전략 설계

In [26]:
# ============================================================
# 12. 24개 항목 판정 전략 초안
# ============================================================
#
# 주의:
# 아래 그룹은 "법적 정의"가 아니라
# 구현 방법을 설계하기 위한 엔지니어링 분류다.
#
# 이후 항목표 + 법령 + dev 사례를 확인하면서 수정한다.
# ============================================================


STRATEGY_GROUPS = {

    # --------------------------------------------------------
    # A. 참가자격 / 지역 / 실적 제한
    # --------------------------------------------------------
    "qualification_restriction": [
        "v1",   # 참가자격 특정기관 제한
        "v2",   # 고시금액 미만 실적제한
        "v3",   # 실적제한 1배수 이상
        "v4",   # 고시금액 이상 특정기관, 특정실적
        "v5",   # 고시금액 이상 지역제한
        "v6",   # 고시금액 미만 지역제한 시,군,구
        "v7",   # 고시금액 미만 지역제한 인접 확대
        "v8",   # 중복제한 (실적+지역)
    ],

    # --------------------------------------------------------
    # B. 물품 / 중소기업 관련
    # --------------------------------------------------------
    "product_sme": [
        "v9",
        "v10",
        "v11",
        "v12",
        "v13",
        "v14",
        "v15",
        "v16",
        "v17",
        "v18",
        "v19",
    ],

    # --------------------------------------------------------
    # C. SW / 공동수급 / 현장설명회
    # --------------------------------------------------------
    "special_conditions": [
        "v20",
        "v21",
        "v22",
        "v23",
    ],

    # --------------------------------------------------------
    # D. 공고 내용 ↔ 나라장터 meta 비교
    # --------------------------------------------------------
    "consistency": [
        "v24",
    ],
}


# ============================================================
# 세부 판정 특성
# ============================================================

ABSENCE_ITEMS = {
    "v10", "v11", "v16", "v18", "v20"
}

AMOUNT_RELATED = {
    "v2", "v3", "v4", "v5", "v6", "v7",
    "v14", "v15", "v16", "v17", "v18",
    "v20"
}

REGION_RELATED = {
    "v5", "v6", "v7", "v8"
}

META_IMPORTANT = {
    "v2", "v3", "v4", "v5", "v6", "v7", "v8",
    "v10", "v11",
    "v13", "v14", "v15", "v16", "v17", "v18",
    "v20", "v22", "v23",
    "v24"
}

DOC_META_COMPARISON = {
    "v24"
}


# ============================================================
# 전략표 생성
# ============================================================

strategy_rows = []

for v in v_cols:

    group = None

    for group_name, group_items in STRATEGY_GROUPS.items():
        if v in group_items:
            group = group_name
            break

    strategy_rows.append({

        "항목": v,

        "항목명":
            items[v].get("항목명"),

        "그룹":
            group,

        "부재탐지":
            v in ABSENCE_ITEMS,

        "금액조건":
            v in AMOUNT_RELATED,

        "지역조건":
            v in REGION_RELATED,

        "meta중요":
            v in META_IMPORTANT,

        "docs-meta비교":
            v in DOC_META_COMPARISON,

        "dev양성수":
            int(labels[v].sum()),
    })


strategy_table = pd.DataFrame(strategy_rows)

display(strategy_table)


# ============================================================
# 그룹별 항목 출력
# ============================================================

print("\n" + "=" * 70)
print("판정 전략 그룹")
print("=" * 70)

for group_name, group_items in STRATEGY_GROUPS.items():

    print(f"\n[{group_name}]")

    for v in group_items:
        print(
            f"  {v:>3}  "
            f"{items[v].get('항목명')}"
        )


# ============================================================
# 검증
# ============================================================

all_grouped_items = []

for group_items in STRATEGY_GROUPS.values():
    all_grouped_items.extend(group_items)

assert len(all_grouped_items) == 24
assert len(set(all_grouped_items)) == 24
assert set(all_grouped_items) == set(v_cols)

assert ABSENCE_ITEMS == {
    v for v in v_cols
    if items[v].get("부재탐지") is True
}

print("\n[검증 완료]")
print("24개 항목이 중복/누락 없이 분류되었습니다.")
print("부재탐지 항목도 항목표.json과 일치합니다.")

,항목,항목명,그룹,부재탐지,금액조건,지역조건,meta중요,docs-meta비교,dev양성수
0,v1,참가자격 특정기관 제한,qualification_restriction,False,False,False,False,False,7
1,v2,고시금액 미만 실적제한,qualification_restriction,False,True,False,True,False,7
2,v3,실적제한 1배수 이상,qualification_restriction,False,True,False,True,False,8
3,v4,"고시금액 이상 특정기관, 특정실적",qualification_restriction,False,True,False,True,False,6
4,v5,고시금액 이상 지역제한,qualification_restriction,False,True,True,True,False,7
5,v6,"고시금액 미만 지역제한 시,군,구",qualification_restriction,False,True,True,True,False,6
6,v7,고시금액 미만 지역제한 인접 확대,qualification_restriction,False,True,True,True,False,7
7,v8,중복제한 (실적+지역),qualification_restriction,False,False,True,True,False,6
8,v9,과업지시서 특정 모델명 명시,product_sme,False,False,False,False,False,6
9,v10,중기간 경쟁제품 입찰 직생 없음,product_sme,True,False,False,True,False,7



판정 전략 그룹

[qualification_restriction]
   v1  참가자격 특정기관 제한
   v2  고시금액 미만 실적제한
   v3  실적제한 1배수 이상
   v4  고시금액 이상 특정기관, 특정실적
   v5  고시금액 이상 지역제한
   v6  고시금액 미만 지역제한 시,군,구
   v7  고시금액 미만 지역제한 인접 확대
   v8  중복제한 (실적+지역)

[product_sme]
   v9  과업지시서 특정 모델명 명시
  v10  중기간 경쟁제품 입찰 직생 없음
  v11  중기간 경쟁제품 입찰 중소 없음
  v12  일반제품 직생 제한
  v13  중기간 경쟁제품 소기업, 소상공인 제한
  v14  고시금액 이상 일반물품 중소기업 제한
  v15  1억 이상- 고시금액미만 소기업 제한
  v16  1억 이상- 고시금액미만 중소기업 제한 없음
  v17  1억원 미만 일반물품 중소기업 제한
  v18  1억원 미만 일반물품 소기업 제한 없음
  v19  물품공급 확약서 입찰 시 제출

[special_conditions]
  v20  입찰참가자격 (SW)
  v21  공동 5% (10%)
  v22  현장설명회 참석업체 (자격으로 제한 여부) * 계약방법 협상만 적용
  v23  현장설명회 (공고기간) * 계약방법 협상 + 계약법 지방만 적용

[consistency]
  v24  공고서와 나라장터 입력값 상이

[검증 완료]
24개 항목이 중복/누락 없이 분류되었습니다.
부재탐지 항목도 항목표.json과 일치합니다.


In [27]:
# ============================================================
# 12. 24개 항목 분류(그룹) 검증 로직 - 중복/누락 체크
# ============================================================

# 원하시는 그룹 정의 (예시 - 실제 분류 기준으로 교체)
GROUPS = {
    "qualification_restriction": ["v1", "v2", "v3", "v4", "v5", "v6", "v7", "v8"],
    "product_sme": ["v9", "v10", "v11", "v12", "v13", "v14", "v15", "v16", "v17", "v18", "v19"],
    "special_conditions": ["v20", "v21", "v22", "v23"],
    "consistency": ["v24"],
}

all_grouped = [v for group in GROUPS.values() for v in group]

# 1) 중복 체크: 같은 항목이 두 그룹 이상에 들어갔는지
duplicates = [v for v in set(all_grouped) if all_grouped.count(v) > 1]
assert not duplicates, f"중복 분류된 항목: {duplicates}"

# 2) 누락 체크: v_cols에는 있는데 어떤 그룹에도 없는 항목
missing = set(v_cols) - set(all_grouped)
assert not missing, f"그룹 분류에서 누락된 항목: {missing}"

# 3) 초과 체크: 그룹에는 있는데 실제 v_cols에 없는 잘못된 항목명
extra = set(all_grouped) - set(v_cols)
assert not extra, f"존재하지 않는 항목이 그룹에 포함됨: {extra}"

print("\n[12] 그룹 분류 검증 완료 (중복/누락/오기 없음)")

# 4) 부재탐지 항목표.json과 실제 items 값 일치 여부까지 검증하려면:
if isinstance(items, dict):
    for v in v_cols:
        expected = items[v].get("부재탐지")
        # 여기서 expected와 별도로 유지하고 있는 GROUPS 기준 부재탐지 라벨이 있다면 비교
        # 현재는 items[v] 자체가 소스이므로 별도 비교 대상이 없다면 이 블록은 생략 가능
        pass


[12] 그룹 분류 검증 완료 (중복/누락/오기 없음)
